# Исторический код автора

Снимок исходной исследовательской логики. Outputs, attachments и метаданные ядра очищены.
Это не основной маршрут запуска; исходная закрытая Excel-выгрузка не включена в проект.
Исторические обозначения Fine–Gray могут быть методологически неточными: см. docs/methods.md.
Для текущего проекта используйте notebooks/01_prepare_data.ipynb и 02_survival_analysis.ipynb.


In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "configs/project.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Откройте ноутбук внутри cardio-oncology-survival-analysis")
sys.path.insert(0, str(ROOT / "src"))
from cardio_oncology.paths import project_paths
P = project_paths(ROOT)
P.run.mkdir(parents=True, exist_ok=True)
P.processed.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = P.run
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / ".matplotlib"))


# Подготовка данных: Кардиоваскулярные осложнения при РТК (v2)
**Исправления относительно v1:**
- Нормализация имён столбцов перенесена в самое начало секции 3 (9 столбцов имели trailing/leading пробелы)
- Два столбца `Таргет` — не дубли, а разные переменные: переименованы корректно
- `parse_date_mixed` расширен: обрабатывает суффикс `'г'` и пробелы-разделители в датах

## 0. Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
from scipy.stats import mannwhitneyu, chi2_contingency

warnings.filterwarnings('ignore')
sns.set(style='whitegrid')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

FILE_PATH   = 'Анализ_РТК_с_ССО__1_.xlsx'
SHEET       = 'Основная таблица'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Загрузка данных

In [ ]:
df_raw = pd.read_excel(FILE_PATH, sheet_name=SHEET)
print(f'Исходный датасет: {df_raw.shape[0]} пациентов × {df_raw.shape[1]} столбцов')
df_raw.head(3)

## 2. Технический аудит

In [ ]:
miss     = df_raw.isnull().sum()
miss_pct = (miss / len(df_raw) * 100).round(1)
audit    = pd.DataFrame({'missing_n': miss, 'missing_%': miss_pct, 'dtype': df_raw.dtypes})
audit    = audit[audit['missing_n'] > 0].sort_values('missing_%', ascending=False)
print(f'Столбцов с пропусками: {len(audit)}')
display(audit)

In [ ]:
empty_cols_raw = [c for c in df_raw.columns if df_raw[c].isna().all()]
print(f'Полностью пустых столбцов: {len(empty_cols_raw)}')

space_cols = [c for c in df_raw.columns if c != c.strip()]
print(f'\nСтолбцов с пробелами в имени: {len(space_cols)}')
for c in space_cols:
    print(f'  {repr(c)}')

cardiotox_col = 'Кардиотокс(1-Да,0-Нет)'
vc = df_raw[cardiotox_col].value_counts()
print(f'\nКардиотоксичность: Есть={vc.get(1,0)} ({vc.get(1,0)/len(df_raw)*100:.1f}%), '
      f'Нет={vc.get(0,0)} ({vc.get(0,0)/len(df_raw)*100:.1f}%)')
print('→ Баланс классов ~52/48% — undersampling не нужен.')

## 3. Нормализация имён, удаление пустых столбцов и дублей

**Порядок важен:** нормализацию имён делаем ПЕРВОЙ — 9 столбцов имеют ведущие/хвостовые пробелы.
Если сделать её позже, любое обращение к этим столбцам до нормализации даст `KeyError`.

In [ ]:
df = df_raw.copy()

# ── ШАГ 1: Нормализация имён столбцов — ОБЯЗАТЕЛЬНО ПЕРВЫМ ──────────────────
# Убирает ведущие/хвостовые пробелы и сжимает внутренние пробелы.
# Затронутые столбцы: 'дата смерти ', 'СРБ на момент начала заболевания ',
# 'Платиносодержащие препараты 0-нет, 1-да ', ' рэа  свежие' и ещё 5 штук.
df.columns = df.columns.str.strip().str.replace(r'\s+', ' ', regex=True)
print('Имена столбцов нормализованы.')

# ── ШАГ 2: Удаление полностью пустых столбцов ────────────────────────────────
empty_cols = [c for c in df.columns if df[c].isna().all()]
df = df.drop(columns=empty_cols)
print(f'Удалено {len(empty_cols)} пустых столбцов. Осталось: {df.shape[1]}')

# ── ШАГ 3: Stage.1 — настоящий дубль Stage ───────────────────────────────────
if 'Stage.1' in df.columns:
    # Используем .equals() вместо ==, он корректно обрабатывает NaN
    if df['Stage'].equals(df['Stage.1']):
        df = df.drop(columns=['Stage.1'])
        print('Stage.1 удалён (подтверждённый дубль Stage).')
    else:
        print('ВНИМАНИЕ: Stage и Stage.1 различаются — оставляем оба для проверки!')

# ── ШАГ 4: Два столбца «Таргет» — это НЕ дубли, а разные переменные ──────────
# После нормализации имён они называются:
#   'Таргет(1-Бев,2-Рег,3-Цет/Пан,4-Комби)'    — бинарный флаг (0=нет, 1=Бевацизумаб)
#   'Таргет(1-Бев,2-Рег,3-Цет/Пан,4-Комби).1'  — полный тип терапии (0/1/3/4/99)
# 27 строк реально различаются → переименовываем оба в понятные имена.
col_bev_flag = 'Таргет(1-Бев,2-Рег,3-Цет/Пан,4-Комби)'
col_tgt_dup  = 'Таргет(1-Бев,2-Рег,3-Цет/Пан,4-Комби).1'

if col_bev_flag in df.columns:
    df = df.rename(columns={col_bev_flag: 'Бевацизумаб_флаг'})
    print(f'Бевацизумаб_флаг (0=нет, 1=да): {df["Бевацизумаб_флаг"].value_counts().to_dict()}')

if col_tgt_dup in df.columns:
    df = df.drop(columns=[col_tgt_dup])

## 4. Даты и возраст

In [ ]:
def parse_date_mixed(val):
    """
    Парсит дату из «грязного» поля датасета.
    Обрабатывает:
      - уже datetime объекты (pandas разобрал при загрузке)
      - строки с точками: '27.09.2024', '10.01.24'
      - строки с пробелами-разделителями: '23 11 2023'
      - суффикс «г»: '25.11.2022г', '17.05.23г'
      - текст с датой в начале: '18.02.2021 C18 ...', '27.07.2025  G93.6'
      - невозможные даты ('11.13.2025') и медицинские тексты → NaT
    """
    if pd.isnull(val):
        return pd.NaT
    if isinstance(val, datetime):
        return val

    s = str(val).strip()
    # Берём первый токен (до первого пробела после даты)
    token = s.split()[0] if s else ''
    # Убираем суффикс «г» (русская традиция: «2022г»)
    token = token.rstrip('г').strip('.,;')

    # Пробуем стандартные форматы с точкой
    for fmt in ('%d.%m.%Y', '%d.%m.%y', '%Y-%m-%d', '%d/%m/%Y'):
        try:
            return datetime.strptime(token, fmt)
        except ValueError:
            continue

    # Пробуем формат с пробелами-разделителями: '23 11 2023'
    parts = s.split()
    if len(parts) >= 3:
        candidate = '.'.join(parts[:3])
        for fmt in ('%d.%m.%Y', '%d.%m.%y'):
            try:
                return datetime.strptime(candidate, fmt)
            except ValueError:
                continue

    return pd.NaT

In [ ]:
# ── 4.1 Дата рождения ────────────────────────────────────────────────────────
df['dob'] = df['Дата рождения'].apply(parse_date_mixed)
n_ok = df['dob'].notna().sum()
print(f'Дата рождения: {n_ok} / {len(df)} распознано')
if n_ok < len(df):
    print('  Не распознано:', df.loc[df['dob'].isna(), 'Дата рождения'].values)

# ── 4.2 Дата выявления заболевания ───────────────────────────────────────────
df['diag_date'] = df['Дата выявления заболевания'].apply(parse_date_mixed)
n_ok_d = df['diag_date'].notna().sum()
print(f'\nДата диагноза: {n_ok_d} / {len(df)} распознано')
if n_ok_d < len(df):
    print('  Не распознано (оставлено NaT):')
    print(df.loc[df['diag_date'].isna(), 'Дата выявления заболевания'].values)

# ── 4.3 Дата смерти (теперь без trailing space — нормализовано в шаге 3) ─────
df['death_date'] = df['дата смерти'].apply(parse_date_mixed)
print(f'\nДата смерти: {df["death_date"].notna().sum()} умерших')

# ── 4.4 Возраст на момент выявления ──────────────────────────────────────────
df['age'] = ((df['diag_date'] - df['dob']).dt.days / 365.25).round(1)

outliers = df[(df['age'] < 18) | (df['age'] > 100)]
if len(outliers):
    print('\nПодозрительный возраст (< 18 или > 100):')
    display(outliers[['Номер пациента', 'Дата рождения', 'Дата выявления заболевания', 'age']])

print('\nВозраст:')
print(df['age'].describe().round(1))

## 5. Стандартизация пола

In [ ]:
sex_map  = {'м': 1, 'М': 1, 'ж': 0, 'Ж': 0}
df['sex'] = df['пол'].map(sex_map)
print('Пол (1=мужчина, 0=женщина):')
print(df['sex'].value_counts(dropna=False))

## 6. Временна́я переменная и статус цензурирования

In [ ]:
df['time_days']       = pd.to_numeric(df['Дней до кардиотоксичности 0- нет даных'], errors='coerce')
df['event_cardiotox'] = pd.to_numeric(df['Кардиотокс(1-Да,0-Нет)'],       errors='coerce').astype('Int64')
df['event_death']     = pd.to_numeric(df['Общая смертность смерть'],        errors='coerce').astype('Int64')
df['event_cv_death']  = pd.to_numeric(df['Сердечно-сосудистая смерть'],     errors='coerce').astype('Int64')

# competing_status для Fine–Gray:
#   0 = цензура (жив без ССО)
#   1 = ССО (интересующее событие)
#   2 = смерть без ССО (конкурирующее событие)
def competing_status(row):
    if row['event_cardiotox'] == 1: return 1
    if row['event_death']     == 1: return 2
    return 0

df['competing_status'] = df.apply(competing_status, axis=1)

print('Время до события (дни):')
print(df['time_days'].describe().round(0))
print()
print('competing_status:')
print(df['competing_status'].value_counts().rename({0:'цензура', 1:'ССО', 2:'смерть без ССО'}))

## 7. Составные исходы (composite endpoints)

In [ ]:
SSO_ACS          = ['ИмспST', 'ИмбпсST', 'Нестабильная стенокардия']
SSO_RHYTHM_MAJOR = ['ФП', 'ЖТ', 'ФЖ', 'полная AV-блокада']
SSO_RHYTHM_MINOR = ['Предсердные экстрасистолии', 'замедление A-V проведения',
                    'ЖЭС', 'Миграция водителя ритма', 'синусовая тахикардия', 'Брадикардия синусовая']
SSO_THROMBO_HF   = ['Декомпенсация ранее существующей СН',
                    'Системные артериальные эмболии', 'Венозные тромбо эмболии']
SSO_OTHER        = ['Развитие de novo ССЗ']
ALL_SSO_COLS     = SSO_ACS + SSO_RHYTHM_MAJOR + SSO_RHYTHM_MINOR + SSO_THROMBO_HF + SSO_OTHER

for c in ALL_SSO_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(int)

df['sso_any']          = (df[ALL_SSO_COLS].sum(axis=1) > 0).astype(int)
df['sso_acs']          = (df[SSO_ACS].sum(axis=1)          > 0).astype(int)
df['sso_rhythm_major'] = (df[SSO_RHYTHM_MAJOR].sum(axis=1)  > 0).astype(int)
df['sso_rhythm_minor'] = (df[SSO_RHYTHM_MINOR].sum(axis=1)  > 0).astype(int)
df['sso_thrombo_hf']   = (df[SSO_THROMBO_HF].sum(axis=1)   > 0).astype(int)

sso_summary = pd.DataFrame({
    'Группа': ['Любое ССО', 'ОКС', 'Значимые аритмии', 'Незначимые аритмии', 'Тромбоз/ХСН'],
    'Флаг':   ['sso_any', 'sso_acs', 'sso_rhythm_major', 'sso_rhythm_minor', 'sso_thrombo_hf'],
    'n':      [df[f].sum() for f in ['sso_any','sso_acs','sso_rhythm_major','sso_rhythm_minor','sso_thrombo_hf']],
    '%':      [(df[f].mean()*100).round(1) for f in ['sso_any','sso_acs','sso_rhythm_major','sso_rhythm_minor','sso_thrombo_hf']]
})
display(sso_summary)

print('\nЧастота отдельных событий:')
ind = df[ALL_SSO_COLS].sum().sort_values(ascending=False)
display(pd.DataFrame({'n': ind, '%': (ind/len(df)*100).round(1)}))

## 8. QTc: конвертация с→мс и ΔQTc

In [ ]:
def clean_qtc(series):
    """
    Унифицирует QTc столбец:
    - Заменяет запятую на точку ('0,38' → 0.38)
    - Значения < 1 → в секундах, умножаем на 1000 → мс
    - Значения вне [250, 700] мс → физически невозможны → NaN
    """
    r = (series.astype(str).str.strip()
               .str.replace(',', '.', regex=False)
               .pipe(pd.to_numeric, errors='coerce'))
    r = r.where(r >= 1, r * 1000)   # секунды → мс
    r = r.where((r >= 250) & (r <= 700))  # вне диапазона → NaN
    return r.round(0)

df['qtc_start_ms'] = clean_qtc(df['QTc начало в секундах'])
df['qtc_last_ms']  = clean_qtc(df['QTc последнее в секундах'])
df['delta_qtc_ms'] = df['qtc_last_ms'] - df['qtc_start_ms']

# Клинические пороги
qtc_thresh = df['sex'].map({1: 440, 0: 460}).fillna(450)
df['qtc_start_prolonged'] = (df['qtc_start_ms'] >= qtc_thresh).astype('Int64')
df['qtc_last_prolonged']  = (df['qtc_last_ms']  >= qtc_thresh).astype('Int64')
df['delta_qtc_gt30']      = (df['delta_qtc_ms'] >= 30).astype('Int64')
df['delta_qtc_gt60']      = (df['delta_qtc_ms'] >= 60).astype('Int64')

print(f'QTc начало (мс): {df["qtc_start_ms"].describe()[["count","mean","min","max"]].round(1).to_dict()}')
print(f'QTc послед. (мс): {df["qtc_last_ms"].describe()[["count","mean","min","max"]].round(1).to_dict()}')
print(f'ΔQTc (мс):        {df["delta_qtc_ms"].describe()[["count","mean","min","max"]].round(1).to_dict()}')
print(f'\nУдлинённый QTc исходно:  {df["qtc_start_prolonged"].sum()}')
print(f'ΔQTc > 30 мс:            {df["delta_qtc_gt30"].sum()}')
print(f'ΔQTc > 60 мс:            {df["delta_qtc_gt60"].sum()}')
print(f'Пропуски QTc:            {df["qtc_start_ms"].isna().sum()} / {len(df)}')

## 9. Очистка числовых предикторов

In [ ]:
def clean_numeric_ru(series, zero_as_nan=False, lower_bound=None, upper_bound=None):
    """Очистка числового столбца: запятые→точки, нечисловые→NaN, опциональные границы."""
    r = (series.astype(str).str.strip()
               .str.replace(',', '.', regex=False)
               .pipe(pd.to_numeric, errors='coerce'))
    if zero_as_nan:    r = r.replace(0, np.nan)
    if lower_bound is not None: r = r.where(r >= lower_bound)
    if upper_bound is not None: r = r.where(r <= upper_bound)
    return r

def winsorize_col(series, lower_q=0.025, upper_q=0.975):
    """Ограничение выбросов на уровне квантилей (winsorizing)."""
    lo = series.quantile(lower_q)
    hi = series.quantile(upper_q)
    return series.clip(lower=lo, upper=hi)

In [ ]:
# ИМТ: запятые в строках + один физически невозможный 0
df['bmi']           = clean_numeric_ru(df['ИМТ'], zero_as_nan=True, lower_bound=10, upper_bound=60)

# ЧСС
df['hr']            = clean_numeric_ru(df['ЧСС'], lower_bound=20, upper_bound=250)

# СОЭ начало
df['esr_start']     = clean_numeric_ru(df['соэ начало'], lower_bound=1, upper_bound=120)

# Гемоглобин начало
df['hgb_start']     = clean_numeric_ru(df['гемоглобин в начале заболевания'], lower_bound=50, upper_bound=200)

# Глюкоза начало
df['glucose_start'] = clean_numeric_ru(df['глюоза на момент начала з-я'],
                                       zero_as_nan=True, lower_bound=2, upper_bound=35)

# ФВ ЛЖ (%)
df['lvef']          = clean_numeric_ru(df['ФВ %'], lower_bound=10, upper_bound=85)

# Размер левого предсердия (мм)
df['la_size']       = clean_numeric_ru(df['Размер левого передсердия'], lower_bound=20, upper_bound=65)

# СРБ начало — имя нормализовано: теперь без trailing space
df['crp_start']     = clean_numeric_ru(df['СРБ на момент начала заболевания'], lower_bound=0)

# Лейкоциты начало
df['wbc_start']     = clean_numeric_ru(df['лейоциты на начало'], lower_bound=1, upper_bound=50)

# Мочевая кислота начало
df['uric_acid']     = clean_numeric_ru(df['мочевая кислота на момент началааболевания'],
                                       zero_as_nan=True, lower_bound=50, upper_bound=1200)

# Холестерин начало
df['chol_start']    = clean_numeric_ru(df['холестерин нач'], zero_as_nan=True, lower_bound=1, upper_bound=15)

# Курсы ХТ
df['n_courses']     = clean_numeric_ru(df['Курсов(кол-во)'])

# Сводка
num_cols_map = {
    'bmi': 'ИМТ', 'hr': 'ЧСС', 'esr_start': 'СОЭ нач.',
    'hgb_start': 'Hgb нач.', 'glucose_start': 'Глюкоза нач.', 'lvef': 'ФВ ЛЖ',
    'la_size': 'Размер ЛП', 'crp_start': 'СРБ нач.', 'wbc_start': 'Лейк. нач.',
    'n_courses': 'Курсов ХТ'
}
for new, label in num_cols_map.items():
    n = df[new].notna().sum()
    print(f'  {label:<20} {n:>3}/{len(df)}  '
          f'[{df[new].min():.1f} – {df[new].max():.1f}]')

In [ ]:
# Winsorizing — те же столбцы, что в ОКС-исследовании
# WINSORIZE_COLS = ['hr', 'esr_start', 'crp_start', 'uric_acid',
#                   'wbc_start', 'hgb_start', 'glucose_start',
#                   'qtc_start_ms', 'qtc_last_ms', 'delta_qtc_ms']

WINSORIZE_COLS = [
    'esr_start',       
    'crp_start',       
    'uric_acid',       
    'wbc_start',       
    'glucose_start',   
]

print('Winsorizing (квантили 2.5% / 97.5%):')
for col in WINSORIZE_COLS:
    if col in df.columns and df[col].notna().sum() > 10:
        lo_before, hi_before = df[col].min(), df[col].max()
        df[col] = winsorize_col(df[col])
        lo_after,  hi_after  = df[col].min(), df[col].max()
        changed = (lo_before != lo_after) or (hi_before != hi_after)
        if changed:
            print(f'  {col:<20} {lo_before:.1f}–{hi_before:.1f}  →  {lo_after:.1f}–{hi_after:.1f}')

In [ ]:
# Клинические бинарные флаги
df['lvef_reduced'] = (df['lvef'] < 50).astype('Int64')           # сниженная ФВ
hgb_thresh         = df['sex'].map({1: 130, 0: 120}).fillna(125)  # норма Hgb по полу
df['anemia']       = (df['hgb_start'] < hgb_thresh).astype('Int64')

print(f'Сниженная ФВ (<50%): {df["lvef_reduced"].sum()} ({df["lvef_reduced"].mean()*100:.1f}%)')
print(f'Анемия на старте:    {df["anemia"].sum()} ({df["anemia"].mean()*100:.1f}%)')

## 10. Кодирование категориальных переменных

In [ ]:
# ── Стадия РТК ───────────────────────────────────────────────────────────────
df['stage'] = pd.to_numeric(df['Stage'], errors='coerce')
df['stage'] = df['stage'].where(df['stage'] > 0)   # -1, 0 → NaN (неизвестна)
df['stage_advanced'] = (df['stage'] >= 3).astype('Int64')  # III–IV vs I–II

print('Стадия:')
print(df['stage'].value_counts(dropna=False))
print(f'Поздняя стадия (III–IV): {df["stage_advanced"].sum()}')

In [ ]:
# ── Схема химиотерапии ────────────────────────────────────────────────────────
# 0=нет ХТ, 1=FOLFOX, 2=FOLFIRI, 3=XELOX, 4=DeGramont, 5=Другое
SCHEME_COL = 'Схема(1-FOLFOX,2-FOLFIRI,3-XELOX,4-DeGramont,5-Другое)'
df['cht_scheme'] = pd.to_numeric(df[SCHEME_COL], errors='coerce').astype('Int64')
SCHEME_LABELS = {0:'Нет ХТ', 1:'FOLFOX', 2:'FOLFIRI', 3:'XELOX', 4:'DeGramont', 5:'Другое'}
df['cht_scheme_label'] = df['cht_scheme'].map(SCHEME_LABELS)

# FOLFOX и XELOX — платиносодержащие + 5-ФУ/капецитабин → наибольший ССР
df['cht_high_cv_risk'] = df['cht_scheme'].isin([1, 3]).astype('Int64')

print('Схема ХТ:')
print(df['cht_scheme_label'].value_counts(dropna=False))

In [ ]:
# ── Таргетная терапия (используем оба переименованных столбца) ────────────────
#   Бевацизумаб_флаг  : 0=нет, 1=получал бевацизумаб
#   ТаргетТерапия_тип : 0=нет, 1=Бев, 3=Цет/Пан, 4=Комби, 99=НД
df['bevacizumab']    = pd.to_numeric(df['Бевацизумаб_флаг'],    errors='coerce').astype('Int64')

print(f'Бевацизумаб (флаг): {df["bevacizumab"].sum()} ({df["bevacizumab"].mean()*100:.1f}%)')


# ── Платиносодержащие ─────────────────────────────────────────────────────────
# Имя нормализовано: 'Платиносодержащие препараты 0-нет, 1-да' (trailing space убран)
df['platinum'] = pd.to_numeric(df['Платиносодержащие препараты 0-нет, 1-да'],
                               errors='coerce').astype('Int64')
print(f'\nПлатиносодержащие: {df["platinum"].sum()} ({df["platinum"].mean()*100:.1f}%)')

# ── Гистология ────────────────────────────────────────────────────────────────
df['histology'] = pd.to_numeric(df['Гистология (1-G1, 2-G2, 3-G3, 99-НД)'],
                                 errors='coerce')
df['histology'] = df['histology'].where(df['histology'] != 99)  # 99 → NaN

# ── ССЗ в анамнезе ────────────────────────────────────────────────────────────
df['cvd_history']      = pd.to_numeric(df['ссз в анамнезе'],           errors='coerce').astype('Int64')
df['rhythm_disturbed'] = pd.to_numeric(df['Ритм(1-Аритм/Тах,0-Норм)'], errors='coerce').astype('Int64')

# ── Метастазы в печень ────────────────────────────────────────────────────────
df['mets_liver'] = pd.to_numeric(df['МТС_Печ (0-нет, 1-есть, 99-НД)'], errors='coerce')
df['mets_liver'] = df['mets_liver'].where(df['mets_liver'] != 99)

print(f'ССЗ в анамнезе:  {df["cvd_history"].sum()} ({df["cvd_history"].mean()*100:.1f}%)')
print(f'Нарушение ритма: {df["rhythm_disturbed"].sum()} ({df["rhythm_disturbed"].mean()*100:.1f}%)')

## 11. Итоговый датасет для анализа

In [ ]:
SURVIVAL_VARS = ['time_days', 'event_cardiotox', 'event_death', 'event_cv_death', 'competing_status']

SSO_FLAGS = ['sso_any', 'sso_acs', 'sso_rhythm_major', 'sso_rhythm_minor', 'sso_thrombo_hf']

PREDICTORS = [
    # Демография
    'age', 'sex',
    # Клинический статус
    'hr', 'rhythm_disturbed', 'lvef', 'lvef_reduced', 'la_size',
    # Лабораторные — воспаление
    'esr_start', 'crp_start', 'wbc_start',
    # Лабораторные — метаболизм
    'bmi', 'hgb_start', 'anemia', 'glucose_start', 'chol_start',
    # QTc
    'qtc_start_ms', 'qtc_last_ms', 'delta_qtc_ms',
    'qtc_start_prolonged', 'delta_qtc_gt30', 'delta_qtc_gt60',
    # Онкологический статус
    'stage', 'stage_advanced', 'histology', 'mets_liver',
    # Лечение
    'cht_scheme', 'cht_scheme_label', 'cht_high_cv_risk',
    'n_courses', 'bevacizumab',
    # Сопутствующие заболевания
    'cvd_history',
]

all_cols  = ['Номер пациента'] + SURVIVAL_VARS + SSO_FLAGS + PREDICTORS + ALL_SSO_COLS
all_cols  = list(dict.fromkeys(c for c in all_cols if c in df.columns))
df_final  = df[all_cols].copy()

COLS_TO_DROP = [
    'platinum',     # дублирует cht_high_cv_risk: из 257 пациентов
                    # только 1 имеет platinum=1 при cht_high_cv_risk=0
]

# Проверяем что столбцы есть перед удалением
existing = [c for c in COLS_TO_DROP if c in df_final.columns]
missing  = [c for c in COLS_TO_DROP if c not in df_final.columns]

if missing:
    print(f'Не найдены (уже удалены?): {missing}')

df_final = df_final.drop(columns=existing)

print(f'Финальный датасет: {df_final.shape[0]} пациентов × {df_final.shape[1]} признаков')
display(df_final.head(3))

In [ ]:
# Проверка пропусков в ключевых переменных
key_cols = ['age', 'sex', 'hr', 'lvef', 'bmi', 'hgb_start', 'glucose_start',
            'stage', 'n_courses', 'time_days', 'event_cardiotox']
miss_report = pd.DataFrame({
    'missing_n': df_final[key_cols].isnull().sum(),
    'missing_%': (df_final[key_cols].isnull().sum() / len(df_final) * 100).round(1)
})
print('Пропуски в ключевых предикторах:')
display(miss_report)
print(f'\nQTc: {df_final["qtc_start_ms"].isna().sum()} пропусков '
      f'({df_final["qtc_start_ms"].isna().mean()*100:.1f}%) — анализ на подвыборке с ЭКГ.')

## 12. Описательная статистика

In [ ]:
# Таблица 1: непрерывные предикторы
cont_preds = ['age', 'hr', 'bmi', 'lvef', 'la_size',
              'hgb_start', 'glucose_start', 'esr_start', 'crp_start',
              'n_courses', 'qtc_start_ms', 'delta_qtc_ms', 'time_days']

desc = df_final[cont_preds].describe(percentiles=[.25, .5, .75]).T
desc = desc[['count', 'mean', 'std', '25%', '50%', '75%', 'min', 'max']].round(2)
desc.columns = ['n', 'Mean', 'SD', 'Q1', 'Median', 'Q3', 'Min', 'Max']
print('Таблица 1 — Непрерывные предикторы:')
display(desc)

In [ ]:
# Таблица 2: сравнение ССО vs без ССО (Mann–Whitney)
rows_mw = []
for col in cont_preds:
    g0 = df_final.loc[df_final['event_cardiotox'] == 0, col].dropna()
    g1 = df_final.loc[df_final['event_cardiotox'] == 1, col].dropna()
    if len(g0) > 1 and len(g1) > 1:
        _, p = mannwhitneyu(g0, g1, alternative='two-sided')
        rows_mw.append({
            'Признак': col,
            'Без ССО Med [Q1–Q3]': f'{g0.median():.1f} [{g0.quantile(.25):.1f}–{g0.quantile(.75):.1f}]',
            'ССО Med [Q1–Q3]':     f'{g1.median():.1f} [{g1.quantile(.25):.1f}–{g1.quantile(.75):.1f}]',
            'p (MW)': f'{p:.3f}'
        })
print('Таблица 2 — Непрерывные предикторы: ССО vs без ССО (Mann–Whitney):')
display(pd.DataFrame(rows_mw))

In [ ]:
# Таблица 3: категориальные предикторы (χ²)
cat_preds = ['sex', 'cvd_history', 'rhythm_disturbed', 'lvef_reduced', 'anemia',
             'stage_advanced', 'bevacizumab', 'cht_high_cv_risk',
             'qtc_start_prolonged', 'delta_qtc_gt30']

rows_cat = []
for col in cat_preds:
    ct = pd.crosstab(df_final['event_cardiotox'], df_final[col])
    if ct.shape == (2, 2):
        _, p, _, _ = chi2_contingency(ct)
        n0 = int(df_final.loc[df_final['event_cardiotox']==0, col].sum())
        n1 = int(df_final.loc[df_final['event_cardiotox']==1, col].sum())
        t0 = (df_final['event_cardiotox']==0).sum()
        t1 = (df_final['event_cardiotox']==1).sum()
        rows_cat.append({'Признак (=1)': col,
                         'Без ССО n (%)': f'{n0} ({n0/t0*100:.1f}%)',
                         'ССО n (%)':     f'{n1} ({n1/t1*100:.1f}%)',
                         'p (χ²)': f'{p:.3f}'})
print('Таблица 3 — Категориальные предикторы: ССО vs без ССО (χ²):')
display(pd.DataFrame(rows_cat))

In [ ]:
# Визуализация: распределение времени наблюдения
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_final.loc[df_final['event_cardiotox']==1, 'time_days'],
             bins=30, alpha=0.7, color='#D65F5F', label='ССО (event=1)')
axes[0].hist(df_final.loc[df_final['event_cardiotox']==0, 'time_days'],
             bins=30, alpha=0.5, color='#4878CF', label='Цензура (event=0)')
axes[0].set_xlabel('Дни до события / цензуры')
axes[0].set_ylabel('Число пациентов')
axes[0].set_title('Распределение времени наблюдения')
axes[0].legend()

df_plot = df_final.dropna(subset=['cht_scheme_label', 'time_days'])
df_plot.boxplot(column='time_days', by='cht_scheme_label', ax=axes[1])
axes[1].set_title('Время наблюдения по схемам ХТ')
axes[1].set_xlabel('Схема ХТ')
axes[1].set_ylabel('Дни')
plt.suptitle('')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Визуализация: QTc
df_qtc = df_final.dropna(subset=['qtc_start_ms', 'qtc_last_ms'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(df_qtc['qtc_start_ms'], ax=axes[0], label='QTc начало', color='#4878CF')
sns.kdeplot(df_qtc['qtc_last_ms'],  ax=axes[0], label='QTc последнее', color='#D65F5F')
axes[0].axvline(440, color='orange', linestyle='--', alpha=0.8, label='Порог 440 мс')
axes[0].set_xlabel('QTc (мс)')
axes[0].set_title('QTc начало vs последнее')
axes[0].legend()

color_ev = df_qtc['event_cardiotox'].map({0: '#4878CF', 1: '#D65F5F'})
axes[1].scatter(df_qtc['qtc_start_ms'], df_qtc['delta_qtc_ms'],
                c=color_ev, alpha=0.5, s=30)
axes[1].axhline(30, color='orange', linestyle='--', label='ΔQTc=30 мс')
axes[1].axhline(60, color='red',    linestyle='--', label='ΔQTc=60 мс')
axes[1].set_xlabel('QTc начало (мс)')
axes[1].set_ylabel('ΔQTc (мс)')
axes[1].set_title('ΔQTc vs QTc начало')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'qtc_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'QTc-подвыборка: n={len(df_qtc)} пациентов')

## 13. Сохранение

In [ ]:
# Основной датасет
df_final.to_csv(OUTPUT_DIR / 'РТК_ССО_подготовленный.csv', index=False, encoding='utf-8-sig')
print(f'Сохранён: РТК_ССО_подготовленный.csv  ({df_final.shape[0]} × {df_final.shape[1]})')

# Словарь переменных
var_dict = pd.DataFrame([
    {'Переменная': 'time_days',           'Тип': 'Время',        'Описание': 'Дни до события / цензуры'},
    {'Переменная': 'event_cardiotox',     'Тип': 'Событие',      'Описание': '1=кардиотоксичность (основной исход)'},
    {'Переменная': 'competing_status',    'Тип': 'Статус',       'Описание': '0=цензура, 1=ССО, 2=смерть без ССО'},
    {'Переменная': 'sso_any',             'Тип': 'Исход',        'Описание': 'Любое ССО-событие'},
    {'Переменная': 'sso_acs',             'Тип': 'Исход',        'Описание': 'ОКС (ИМспST / ИМбпсST / НС)'},
    {'Переменная': 'sso_rhythm_major',    'Тип': 'Исход',        'Описание': 'Значимые аритмии'},
    {'Переменная': 'age',                 'Тип': 'Непрерывная',  'Описание': 'Возраст на момент диагноза (лет)'},
    {'Переменная': 'sex',                 'Тип': 'Бинарная',     'Описание': '1=мужчина, 0=женщина'},
    {'Переменная': 'hr',                  'Тип': 'Непрерывная',  'Описание': 'ЧСС (уд/мин), winsorized'},
    {'Переменная': 'rhythm_disturbed',    'Тип': 'Бинарная',     'Описание': '1=аритмия/тахикардия исходно'},
    {'Переменная': 'lvef',                'Тип': 'Непрерывная',  'Описание': 'ФВ ЛЖ (%)'},
    {'Переменная': 'lvef_reduced',        'Тип': 'Бинарная',     'Описание': '1=ФВ<50%'},
    {'Переменная': 'la_size',             'Тип': 'Непрерывная',  'Описание': 'Размер левого предсердия (мм)'},
    {'Переменная': 'esr_start',           'Тип': 'Непрерывная',  'Описание': 'СОЭ начальное (мм/ч)'},
    {'Переменная': 'hgb_start',           'Тип': 'Непрерывная',  'Описание': 'Гемоглобин начальный (г/л)'},
    {'Переменная': 'anemia',              'Тип': 'Бинарная',     'Описание': '1=анемия (Hb<нормы по полу)'},
    {'Переменная': 'glucose_start',       'Тип': 'Непрерывная',  'Описание': 'Глюкоза начальная (ммоль/л)'},
    {'Переменная': 'bmi',                 'Тип': 'Непрерывная',  'Описание': 'ИМТ (кг/м²)'},
    {'Переменная': 'qtc_start_ms',        'Тип': 'Непрерывная',  'Описание': 'QTc исходный (мс)'},
    {'Переменная': 'delta_qtc_ms',        'Тип': 'Непрерывная',  'Описание': 'ΔQTc = QTc_last − QTc_start (мс)'},
    {'Переменная': 'qtc_start_prolonged', 'Тип': 'Бинарная',     'Описание': '1=QTc исходно удлинён'},
    {'Переменная': 'delta_qtc_gt30',      'Тип': 'Бинарная',     'Описание': '1=ΔQTc≥30 мс'},
    {'Переменная': 'stage',               'Тип': 'Порядковая',   'Описание': 'Стадия РТК (1–4)'},
    {'Переменная': 'stage_advanced',      'Тип': 'Бинарная',     'Описание': '1=III–IV стадия'},
    {'Переменная': 'cht_scheme',          'Тип': 'Категориальная','Описание': '0=нет, 1=FOLFOX, 2=FOLFIRI, 3=XELOX, 4=DeGramont'},
    {'Переменная': 'cht_high_cv_risk',    'Тип': 'Бинарная',     'Описание': '1=FOLFOX/XELOX (высокий ССР)'},
    {'Переменная': 'n_courses',           'Тип': 'Непрерывная',  'Описание': 'Количество курсов ХТ'},
    {'Переменная': 'bevacizumab',         'Тип': 'Бинарная',     'Описание': '1=получал бевацизумаб'},
    {'Переменная': 'cvd_history',         'Тип': 'Бинарная',     'Описание': '1=ССЗ в анамнезе'},
])
var_dict.to_csv(OUTPUT_DIR / 'РТК_ССО_словарь_переменных.csv', index=False, encoding='utf-8-sig')
print('Сохранён: РТК_ССО_словарь_переменных.csv')

print('\n=== Подготовка данных завершена ===')
print(f'  Пациентов:           {len(df_final)}')
print(f'  Признаков в модели:  {len(PREDICTORS)}')
print(f'  Кардиотоксичность:   {df_final["event_cardiotox"].sum()} ({df_final["event_cardiotox"].mean()*100:.1f}%)')
print(f'  Медиана наблюдения:  {df_final["time_days"].median():.0f} дней')
print(f'  Следующий шаг:       Kaplan–Meier + Cox regression (lifelines)')